# Star map: the shipped strategy links


The star map draws gene-gene links with their provenance. Measured links are read from each space's graph at run time. This notebook makes the other shipped part: the links stated by the default-setting runs of the strategies that relate genes to genes, on the shipped tables. The code is `scripts/build_star_edges.py` and the rules that bound each kind of link are in `starplast/star_edges.py`; this notebook calls them, so what is shown is what ships in `starplast/data/star_edges.parquet`.

No strategy or calibration number is changed: each strategy is run with its defaults.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.insert(0, '/media/carruthers/mnt3/claude/repo/starplast/.claude/worktrees/agent-a55c7f316312c8704'); sys.path.insert(0, '/media/carruthers/mnt3/claude/repo/starplast/.claude/worktrees/agent-a55c7f316312c8704/scripts')
import pandas as pd
import build_star_edges as B
from starplast import star_edges as E
pd.DataFrame({'bound': ['MODULE_K', 'SEED_K', 'LABEL_K', 'PARTNER_MAX', 'MAX_EDGES_PER_RUN', 'SHIPPED_SEED_SETS'], 'value': [E.MODULE_K, E.SEED_K, E.LABEL_K, E.PARTNER_MAX, E.MAX_EDGES_PER_RUN, E.SHIPPED_SEED_SETS]})

,bound,value
0,MODULE_K,3
1,SEED_K,3
2,LABEL_K,3
3,PARTNER_MAX,5
4,MAX_EDGES_PER_RUN,20000
5,SHIPPED_SEED_SETS,3


## 1. Run every strategy once per space

Seed-list strategies (20, 25) run once per example seed list.

In [2]:
links, log = B.build(log=lambda m: None)
log

,organism,strategy,run,links,seconds,note
0,Tg,holdout_search,starplast:holdout_search,2270,56.1,"Best of 16 configurations: all permitted, n_neighbors 15, min_dist 0.25, min cluster 20 (leaf). Mean F1 0.303 over the compartment labels, against 0.157 ± 0.008"
1,Tg,consensus_modules,starplast:consensus_modules,3119,19.2,"24 clusterings over 3,000 genes; 14 modules whose members co-cluster in at least 50% of them (1,405 genes). A module is what survives the choice of settings --"
2,Tg,feature_knn,starplast:feature_knn,6000,0.5,"2,000 unlabelled genes called from their 15 nearest labelled genes in 359 permitted measurements (vote >= 30%). Distance is taken over every measurement at once"
3,Tg,cluster_guilt,starplast:cluster_guilt,2140,8.9,"12 clusters are enriched for one compartment label at q < 0.05 and lift >= 2 (hypergeometric, BH-corrected over every cluster x label); their 301 unlabelled mem"
4,Tg,physical_partners,starplast:physical_partners,347,2.5,"271 unlabelled genes with a measured physical partner in xlms + ip_ms are called by their partners' compartment, weighted by crosslink or pulldown count. Each c"
5,Tg,multiplex_modules,starplast:multiplex_modules,20000,180.3,"6 communities that the layers coexpression, cofitness, cotranslation, domain, ip_ms, struct, xlms agree on; 7,822 genes placed in one. 2,000 unlabelled members"
6,Tg,link_prediction,starplast:link_prediction,300,0.5,"A model of what makes a xlms edge -- trained on its 2,842 edges against random non-edges, from 8 other layers, shared partners and measurement similarity -- sco"
7,Tg,attention_correction,starplast:attention_correction,200,0,Top 200 comention_ft pairs by attention-corrected residual: 77% share a compartment label (where both have one); by raw count: 65%; all pairs: 39%. The correcte
8,Tg,unwritten_links,starplast:unwritten_links,500,0,"500 pairs linked in at least 2 independent measurement layers (coexpression, cofitness, cotranslation, ip_ms, struct, xlms) that no abstract or paragraph mentio"
9,Tg,positive_unlabeled,starplast:positive_unlabeled:PM - integral,900,1,"135 positives against the rest of the proteome, 15 bags; the 300 genes that most resemble your list are listed, scored only by models that did not train on them"


## 2. Links per source

In [3]:
links.groupby(['organism', 'source', 'how']).size().rename('links').reset_index()

,organism,source,how,links
0,Pf,attention_correction,pair,200
1,Pf,cluster_guilt,module,2264
2,Pf,consensus_modules,module,5541
3,Pf,feature_knn,label,6000
4,Pf,holdout_search,module,1825
5,Pf,link_prediction,pair,126
6,Pf,multiplex_modules,module,14975
7,Pf,neighbour_space,pair,5125
8,Pf,network_training,pair,300
9,Pf,physical_partners,partner,23


## 3. What a link carries

Three examples: a predicted pair, a module link and a seed link.

In [4]:
pd.concat([links[links['how'] == h].head(1) for h in ('pair', 'module', 'seed')]).T

,33876,0,34876
organism,Tg,Tg,Tg
gene_a,TGME49_261580,TGME49_320230,TGME49_247195
gene_b,TGME49_261250,TGME49_301150,TGME49_285180
kind,inferred,inferred,inferred
group,link_prediction,holdout_search,positive_unlabeled
source,link_prediction,holdout_search,positive_unlabeled
how,pair,module,seed
setting,"layer=xlms, top=300","target=compartment, n_neighbors=15, 50, min_dist=0.0, 0.25, min_cluster_size=20, 50, selection=eom, leaf, sample=3000, …","seed list = 'PM - integral' (135 genes); genes=135 genes, bags=15, top=300"
score,21.44,5.675,4.316
strength,1,1,0.9989


## 4. Write the shipped file

In [5]:
import os
path = B.write(links)
print(path, f'{os.path.getsize(path) / 1e6:.2f} MB', f'{len(links):,} links')

/media/carruthers/mnt3/claude/repo/starplast/.claude/worktrees/agent-a55c7f316312c8704/starplast/data/star_edges.parquet 1.16 MB 87,115 links
